## **GDP**

### **Purpose:**
- Connect to the FRED API
- Graph latest GDP
- Ingest total market cap of S&P 500 data Source: https://ycharts.com/indicators/sp_500_market_cap 
- Graph the S&P500 mkt cap data source to GDP

In [ ]:
# import csv

import csv
import pandas as pd

mktcap = pd.read_csv('total_mktcap.csv')


In [ ]:
mktcap.head()

In [ ]:
mktcap.info()

In [ ]:
mktcap["Value"] = mktcap["Value"].str[:5].astype(float) * 1*10**12
#Note: I had to adjust the month by 1 because the GDP date was one day off in the raw data. This was to make sure a join could work when pulling the two data sources into the same dataframe.
mktcap["Date"] = (pd.to_datetime(mktcap["Date"], format="%B %d, %Y") + pd.DateOffset(months=1)).dt.strftime("%Y-%m-%d")


In [ ]:
mktcap.info()

In [ ]:
mktcap.head()

In [ ]:
# Import Libraries and set parameters for api query
import requests
import json
import pprint
import pandas as pd
import matplotlib.pyplot as plt


# Enter your own API_Key after registering for it from FRED.
with open('/Users/kalebferrer/python/stuff/api_key.txt', 'r') as file:
    api_key = file.readline().strip()

# Set parameters
series_id = [ 'GDP','A191RL1Q225SBEA', 'GDPC1', 'A191RP1Q027SBEA', 'A939RX0Q048SBEA', 'GDPNOW']

file_type = 'json'
base_url_series = 'https://api.stlouisfed.org/fred/series'
base_url_obs = 'https://api.stlouisfed.org/fred/series/observations'

# Create Uniform Resource Location object

for series in series_id:
    # Pull information about series
    url_series = f'{base_url_series}?series_id={series}&api_key={api_key}&file_type={file_type}'
    # Send API request
    response = requests.get(url_series)
    # Convert HTTP request to dictionary using the .json method
    response_dict = response.json()
    
    series_list = response_dict['seriess']
    
    titles = []
    for item in series_list:
        #pprint.pprint(item)
        titles.append(item['title'])
        titles.append(item['units'])
    
    #Pull observations from the series
    url1 = f'{base_url_obs}?series_id={series}&api_key={api_key}&file_type={file_type}'
    response = requests.get(url1)
    response_dict = response.json()
    

    #Graph the observations from the series - WORK ON ADDING TITLE

    obs_list = response_dict['observations']
    obs_df = pd.DataFrame(obs_list)
    obs_df['date'] = pd.to_datetime(obs_df['date'])
    obs_df['value'] = pd.to_numeric(obs_df['value'], errors='coerce')

    if series == 'GDP':
        gdp_df = obs_df

    obs_df.plot(x='date', y='value', kind='line')

    plt.suptitle(titles[0])
    plt.title(titles[1])
    
    plt.show()

    print(titles[0])
    print(titles[1])
    print(obs_df[['date', 'value']].tail(24).sort_values(by='date', ascending=False))

In [ ]:
gdp_df["date"] = gdp_df['date'].astype(str)
gdp_df["JoinKey"] = gdp_df["date"].str[:7]
mktcap['Date'] = mktcap['Date'].astype(str)
mktcap["JoinKey"] = mktcap["Date"].str[:7]

In [ ]:
merged_df = gdp_df.merge(mktcap, on="JoinKey", suffixes=("_gdp_df", "_mktcap")).drop(columns=["JoinKey"])

In [ ]:
merged_df['value'] = merged_df['value'] * 1*10**9

In [ ]:
merged_df['sp500_mktcap_to_gdp'] = merged_df['Value'] / merged_df['value']

In [ ]:
merged_df.plot(x='Date', y='sp500_mktcap_to_gdp', kind='line')

In [ ]:
merged_df["Date"] = merged_df["Date"].astype(str).str[:7]  # Keep only "YYYY-MM"

In [ ]:
merged_df.plot(x='Date', y='sp500_mktcap_to_gdp', kind='line')

In [ ]:
import matplotlib.pyplot as plt

# Set figure size
plt.figure(figsize=(12, 6))

# Convert Date to first 7 characters (YYYY-MM)
merged_df["Date"] = merged_df["Date"].astype(str).str[:7]

# Compute mean and standard deviation
mean_value = merged_df["sp500_mktcap_to_gdp"].mean()
std_dev = merged_df["sp500_mktcap_to_gdp"].std()

# Plot the main data
ax = merged_df.plot(x="Date", y="sp500_mktcap_to_gdp", kind="line", legend=False, ax=plt.gca())

# Add horizontal mean line
plt.axhline(mean_value, color="red", linestyle="--", linewidth=2, label=f"Mean: {mean_value:.2f}")

# Add shaded area for ±1 standard deviation
plt.fill_between(merged_df["Date"], mean_value - std_dev, mean_value + std_dev, 
                 color="gray", alpha=0.3, label=f"±1 Std Dev ({std_dev:.2f})")

# Add shaded area for ±2 standard deviations (lighter shading)
plt.fill_between(merged_df["Date"], mean_value - 2*std_dev, mean_value + 2*std_dev, 
                 color="gray", alpha=0.15, label=f"±2 Std Dev ({2*std_dev:.2f})")

# Customize x-axis tick frequency
ax.set_xticks(merged_df["Date"][::2])  # Adjust step (e.g., every 2nd date)
ax.set_xticklabels(merged_df["Date"][::2], rotation=45)  # Rotate labels

# Labels and title
plt.xlabel("Date (YYYY-MM)")
plt.ylabel("S&P 500 Market Cap to GDP")
plt.title("S&P 500 Market Cap to GDP Over Time")

# Show legend
plt.legend()

# Show the plot
plt.show()


In [185]:
%reset -f